# Workshop 2: Identifying Deception in Diplomacy

In this workshop, you will use scikit-learn to identify deception in the board game [Diplomacy](https://en.wikipedia.org/wiki/Diplomacy_(game)).

**Instructions:**
- Form a pair. You should have just one computer and one person typing at a time.
- We will be using the data from this repository: https://github.com/DenisPeskoff/2020_acl_diplomacy
- If you are curious about the background, see the research paper: https://aclanthology.org/2020.acl-main.353/

**Data files:**
- `mod-train.jsonl` — training data
- `mod-validation.jsonl` — development / validation data
- `mod-test.jsonl` — test data (only use at the very end)

---
## Data Format

Each line in the JSONL files is a JSON object representing one conversation between two players. The key fields are:

| Field | Description |
|-------|-------------|
| `messages` | List of text messages sent between players |
| `sender_labels` | List of `true` (truthful) or `false` (lie) for each message |
| `receiver_labels` | Whether the receiver thought it was truth (`true`), lie (`false`), or didn't label (`"NOANNOTATION"`) |
| `speakers` | Who sent each message (e.g. `"italy"`, `"germany"`) |
| `receivers` | Who received each message |
| `seasons` | Game season (`"Spring"`, `"Fall"`, `"Winter"`) |
| `years` | Game year (`"1901"`, `"1902"`, etc.) |

For Task 1, you only need `messages` and `sender_labels`. The other fields will be used in Tasks 2 and 3.

---
## Task 1: Basic Classifier

Implement a **Logistic Regression** classifier that predicts whether a message is truthful or a lie.

**Configuration:**
- Solver: `saga`
- Penalty: `None`
- All other parameters as defaults

**Hints:**
- In the prework, `twenty_train.data` was a list of strings — you need to build the same thing from the JSONL data.
- In the prework, `twenty_train.target` was a numpy array — you can use `numpy.array(my_list)` to create one.
- Use the same Pipeline pattern from the prework (`CountVectorizer` → `TfidfTransformer` → classifier).

### Step 1: Data Loading

Load the training and development datasets. You need to create:
- A list of message strings (like `twenty_train.data`)
- A numpy array of labels: `1` for truthful, `0` for lie (like `twenty_train.target`)

In [5]:
# TODO: Import the libraries you need
import json
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn import metrics

In [6]:
# TODO: Write a function to load a JSONL file
# Each line is a JSON object (use json.loads)
# Each object has "messages" (list of strings) and "sender_labels" (list of True/False)
# Return a flat list of all messages and a numpy array of labels (1=truth, 0=lie)
def readfile(filename):
    examples_msg = []
    examples_labels = []
    for line in open(filename):
        example = json.loads(line.strip())
        for msg, tlabel in zip(example['messages'], example['sender_labels']):
            examples_msg.append(msg)
            examples_labels.append(tlabel)
    return examples_msg, np.array(examples_labels)

In [8]:
# TODO: Load the training and development data using your function
# Print the number of messages and lies in each to verify
train = readfile('mod-train.jsonl')
dev = readfile('mod-train.jsonl')

In [11]:
print(train[0])

['Germany!\n\nJust the person I want to speak with. I have a somewhat crazy idea that I’ve always wanted to try with I/G, but I’ve never actually convinced the other guy to try it. And, what’s worse, it might make you suspicious of me. \n\nSo...do I suggest it?\n\nI’m thinking that this is a low stakes game, not a tournament or anything, and an interesting and unusual move set might make it more fun? That’s my hope anyway.\n\nWhat is your appetite like for unusual and crazy?', "You've whet my appetite, Italy. What's the suggestion?", '👍', "It seems like there are a lot of ways that could go wrong...I don't see why France would see you approaching/taking Munich--while I do nothing about it--and not immediately feel skittish", 'Yeah, I can’t say I’ve tried it and it works, cause I’ve never tried it or seen it. But how I think it would work is (a) my Spring move looks like an attack on Austria, so it would not be surprising if you did not cover Munich. Then (b) you build two armies, which

### Steps 2-4: Preprocessing, Model Definition, and Training

Build a Pipeline (just like in the prework) that chains:
1. `CountVectorizer` — convert text to word counts
2. `TfidfTransformer` — apply TF-IDF weighting
3. `LogisticRegression` — with `solver='saga'` and `penalty=None`

Then train the pipeline on your training data.

In [9]:
# TODO: Create a Pipeline and train it on the training data
text_clf = Pipeline([('vect', CountVectorizer()),
                     ('tfidf', TfidfTransformer()),
                     ('clf', LogisticRegression(solver='saga', penalty=None))])
text_clf.fit(train[0], train[1])

predicted = text_clf.predict(dev[0])



/opt/anaconda3/envs/dl_env/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


### Step 5: Model Evaluation

Evaluate on the **development** dataset using `metrics.classification_report`.

You should get results that look something like this:

```
              precision    recall  f1-score   support

           0       0.07      0.04      0.05       124
           1       0.96      0.97      0.96      2658

    accuracy                           0.93      2782
   macro avg       0.51      0.51      0.51      2782
weighted avg       0.92      0.93      0.92      2782
```

The key number is the **f1-score for class 0 (lies)**: ~0.05. This is not very good — it's a hard problem!

> **Note:** Accuracy looks high (0.93) because ~95% of messages are truthful. A model that always predicts "truth" would get ~95% accuracy.

In [10]:
# TODO: Predict on the development set and print the classification report
print(np.mean(predicted == dev[1]))
print(metrics.classification_report(dev[1], predicted))
print(metrics.confusion_matrix(dev[1], predicted))

0.9905660377358491
              precision    recall  f1-score   support

       False       0.99      0.80      0.88       523
        True       0.99      1.00      1.00     11243

    accuracy                           0.99     11766
   macro avg       0.99      0.90      0.94     11766
weighted avg       0.99      0.99      0.99     11766

[[  418   105]
 [    6 11237]]


---
## Task 2: Analysis

Before trying to improve the classifier, let's investigate some properties of the dataset.

### 2a. How accurate are people?

The `receiver_labels` field contains whether the receiver of the message thought it was the truth (`true`) or not (`false`). For cases where they did not provide a label (`"NOANNOTATION"`), treat it as `true` (i.e., they assumed the message was truthful).

Using `sender_labels` as the ground truth and `receiver_labels` as the predictions, calculate the **precision** and **recall** of the receivers using `classification_report`.

In [14]:
print(train[1])

[ True  True  True ...  True  True  True]


In [18]:
import json
from sklearn.metrics import classification_report

# 准备一维的平铺列表
sender_labels_flat = []
receiver_labels_flat = []

# 打开 jsonl 文件
with open("mod-train.jsonl", "r", encoding="utf-8") as file:
    for line in file:
        data = json.loads(line)
        
        # 使用 extend() 而不是 append()，这样可以把内部的元素逐个加进去，保持一维列表
        sender_labels_flat.extend(data.get('sender_labels', []))
        receiver_labels_flat.extend(data.get('receiver_labels', []))

# 处理 receiver_labels 中的 "NOANNOTATION"
# 如果是 "NOANNOTATION" 就视为 True；如果已经是布尔值 (True/False)，就保持原样
processed_receiver_labels = [
    True if label == "NOANNOTATION" else bool(label) 
    for label in receiver_labels_flat
]

# 打印分类报告
# sender_labels_flat 是 Ground Truth (真实标签)
# processed_receiver_labels 是 Predictions (人类玩家的预测)
print(classification_report(sender_labels_flat, processed_receiver_labels))

              precision    recall  f1-score   support

       False       0.12      0.12      0.12       523
        True       0.96      0.96      0.96     11243

    accuracy                           0.92     11766
   macro avg       0.54      0.54      0.54     11766
weighted avg       0.92      0.92      0.92     11766



### 2b. Lie distribution across features

For each of the following fields, calculate the **percentage of messages that are lies** for each possible value:
- `speakers`
- `receivers`
- pairs of (`speakers`, `receivers`)
- `seasons`
- `years`

In [0]:
# TODO: Calculate the percentage of lies for each unique value of each field


### 2c. Manual inspection

Read through 20 of the training set messages that are **lies** and 20 that are the **truth**.

Do you notice anything different about them? Write your observations below.

In [0]:
# TODO: Print 20 messages that are lies and 20 that are truths


*Your observations:*

(Write here)

---
## Task 3: Improved Classifier

In Task 1, we only used the message text. Can you add other properties of the data to try to improve the F1-score for lies (class 0)?

**Ideas to try:**
- Add metadata features (speaker, receiver, season, year) alongside the text features
- Engineer new features (message length, punctuation counts, deception-related word counts)
- Tune the `CountVectorizer` (e.g., try bigrams with `ngram_range=(1, 2)`)
- Address class imbalance (e.g., `class_weight='balanced'` in `LogisticRegression`)

**Important:** Experiment on the **development/validation** data. Only evaluate on the **test** data once you have a final approach.

> If you manage to beat the baseline, let your tutor know! They will track the best result and share it with the group.

In [0]:
# TODO: Try to improve the classifier!


In [0]:
# TODO: Once you're happy with your approach, evaluate on the TEST set
